# FFTW C2C Python Sequential

- https://numpy.org/doc/stable/reference/routines.fft.html
- https://hgomersall.github.io/pyFFTW/

In [41]:
#=-----------------------------------------------------------------------=#

## Numpy FFT

In [57]:
import numpy as np
N = 12
a = np.fromfunction( lambda i, j, k: (i*N**2 + j*N  + k + 1)*1E-6,
                    (N, N, N), dtype=np.complex128 )
print('D: ', np.sum(a))

D:  (1.4938559999999999+0j)


In [56]:
import numpy as np
N = 128
print('D: {:.2f}'.format(np.sum(a)))
f = np.fft.fftn(a)
s = np.sum(f)
print('S: {:.2f}'.format(s))

D: 2199024.30+0.00j
S: 2.10-0.00j


In [58]:
import numpy as np
N = 128
print('S: {:.4f}'.format(
    np.sum(
        np.fft.fftn(
            np.fromfunction(
                lambda i, j, k:
                    (i*N**2 + j*N  + k + 1)*1E-6,
                    (N, N, N), dtype=np.complex128 ) ) ) ))

S: 2.0972-0.0000j


In [60]:
%%timeit -n 1 -r 1
import numpy as np
N = 500
f = np.fromfunction( lambda i, j, k: (i*N**2 + j*N  + k + 1)*1E-6,
                     (N, N, N), dtype=np.complex128 )
print('D: {:.2f}'.format( np.sum( f ) ) )
print('S: {:.2f}'.format( np.sum( np.fft.fftn( f ) ) ) )

D: 7812500062.50+0.00j
S: 125.00+0.00j
12.2 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


## FFTW
1D example, random

In [8]:
import numpy
import pyfftw
a = pyfftw.empty_aligned(128, dtype='complex128', n=16)
a[:] = numpy.random.randn(128) + 1j*numpy.random.randn(128)
b = pyfftw.interfaces.numpy_fft.fft(a)
c = numpy.fft.fft(a)
numpy.allclose(b, c)

True

### The above code written differently 

In [12]:
import numpy as np, pyfftw as pf
N = 128
a = pf.empty_aligned(N, dtype=np.complex128, n=16)
a[:] = np.random.randn(128) + 1j*np.random.randn(128)
np.allclose(
    pf.interfaces.numpy_fft.fft(a),
    np.fft.fft(a)
)

True

### Based on the above codes

One-dimensional FFT, random

In [61]:
%%timeit -n 1 -r 1
import numpy as np, pyfftw as pf
N = 128
# n=16 : array aligned to a particular number of bytes in memory
a = pf.empty_aligned(N, dtype=np.complex128, n=16)
a[:] = np.random.randn(128) + 1j*np.random.randn(128)
b = pf.interfaces.numpy_fft.fft(a)
print('S: {:.4f}'.format( np.sum( b ) ) )

S: 45.5328+49.9659j
44.1 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### 3D 128x128x128

Compute the N-dimensional discrete Fourier Transform, function

In [64]:
%%timeit -n 1 -r 1
import numpy as np, pyfftw as pf, time as tm
t0 = tm.time()
N = 128
a = pf.empty_aligned( (N, N, N), dtype=np.complex128 )
a[:,:,:] = np.fromfunction( lambda i, j, k:
    (i*N**2 + j*N  + k + 1)*1E-6, (N, N, N), dtype=np.complex128 )
print('D: {:.2f}'.format( np.sum(a) ) )
b = pf.interfaces.numpy_fft.fftn(a)
c = np.fft.fftn(a)
t1 = tm.time()
print('S: {:.2f}    Ts: {:.4f} s'.format( np.sum( c ), t1-t0 ) )

D: 2199024.30+0.00j
S: 2.10-0.00j    Ts: 0.3367 s
340 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [66]:
%%timeit -n 1 -r 1
import numpy as np, pyfftw as pf, time as tm
t0 = tm.time()
N = 128
a = pf.empty_aligned( (N, N, N), dtype=np.complex128 )
a[:,:,:] = np.fromfunction( lambda i, j, k :
    (i*N**2 + j*N  + k + 1)*1E-6,
    (N, N, N), dtype=np.complex128 )
print('D: {:.2f}'.format( np.sum(a) ) )
b = pf.interfaces.numpy_fft.fftn(a)
c = np.fft.fftn(a)
t1 = tm.time()
print('S: {:.2f}    T: {:.4f} s'.format( np.sum( c ), t1-t0 ) )

D: 2199024.30+0.00j
S: 2.10-0.00j    Ts: 0.3018 s
306 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### 3D 3x3x3, modified function

In [74]:
import numpy as np, pyfftw as pf, time as tm
np.set_printoptions(precision=2)
t0 = tm.time()
N = 3
a = pf.empty_aligned( (N, N, N), dtype=np.complex128 )
a[:,:,:] = np.fromfunction( lambda i, j, k :
    (i*N**2 + j*N  + k + 1),
    (N, N, N), dtype=np.complex128 )
print('D: {:.2f}'.format( np.sum(a) ) )
b = pf.interfaces.numpy_fft.fftn(a)
c = np.fft.fftn(a)
t1 = tm.time()
print('S: {:.2f}    T: {:.4f} s'.format( np.sum( c ), t1-t0 ) )
c

D: 378.00+0.00j
S: 27.00+0.00j    T: 0.0014 s


array([[[ 378.  +0.j  ,  -13.5 +7.79j,  -13.5 -7.79j],
        [ -40.5+23.38j,    0.  +0.j  ,    0.  +0.j  ],
        [ -40.5-23.38j,    0.  +0.j  ,    0.  +0.j  ]],

       [[-121.5+70.15j,    0.  +0.j  ,    0.  +0.j  ],
        [   0.  +0.j  ,    0.  +0.j  ,    0.  +0.j  ],
        [   0.  +0.j  ,    0.  +0.j  ,    0.  +0.j  ]],

       [[-121.5-70.15j,    0.  +0.j  ,    0.  +0.j  ],
        [   0.  +0.j  ,    0.  +0.j  ,    0.  +0.j  ],
        [   0.  +0.j  ,    0.  +0.j  ,    0.  +0.j  ]]])

### 3D 500x500x500

In [69]:
import numpy as np, pyfftw as pf, time as tm
t0 = tm.time()    # <--- time measurement
N = 500
a = pf.empty_aligned( (N, N, N), dtype=np.complex128 )
a[:,:,:] = np.fromfunction( lambda i, j, k :
    (i*N**2 + j*N  + k + 1)*1E-6,
    (N, N, N), dtype=np.complex128 )
print('D: {:.2f}'.format( np.sum(a) ) )
b = pf.interfaces.numpy_fft.fftn(a)
s = np.sum(b)
t1 = tm.time()    # <--- time measurement
print('S: {:.2f}    T: {:.4f} s'.format(s, t1-t0))

D: 7812500062.50+0.00j
S: 125.00-0.00j    T: 19.7202 s


## Runs on an execution node

In [1]:
%%writefile pc2cs.py
import numpy as np, pyfftw as pf, time as tm
t0 = tm.time()    # <--- time measurement
N = 500
a = pf.empty_aligned( (N, N, N), dtype=np.complex128 )
a[:,:,:] = np.fromfunction( lambda i, j, k :
    (i*N**2 + j*N  + k + 1)*1E-6,
    (N, N, N), dtype=np.complex128 )
b = pf.interfaces.numpy_fft.fftn(a)
s = np.sum(b)
t1 = tm.time()    # <--- time measurement
print('S: {:.2f}    T: {:.4f} s'.format(s, t1-t0))

Overwriting pc2cs.py


Copy to /scratch

In [3]:
%%bash
dst=/scratch${PWD#"/prj"}
mkdir -p $dst
cp pc2cs.py $dst

## Batch script

In [2]:
%%writefile pc2cs.srm
#!/bin/bash
# 1,0 UA partitions:
#   cpu_dev  : 20 min.,  1-4  nodes, 1/1   tasks
#   cpu_small: 72 hours, 1-20 nodes, 16/96 tasks
#SBATCH --partition cpu_small  # Select partition
#SBATCH --ntasks=1             # Total tasks
#SBATCH --job-name pc2cs       # Job name
#SBATCH --time=00:01:00        # Limit execution time
#SBATCH --exclusive            # Exclusive acccess to nodes

echo '========================================'
echo '- Job ID:' $SLURM_JOB_ID
echo '- Tasks per node:' $SLURM_NTASKS_PER_NODE
echo '- # of nodes in the job:' $SLURM_JOB_NUM_NODES
echo '- # of tasks:' $SLURM_NTASKS
echo '- Dir from which sbatch was invoked:' ${SLURM_SUBMIT_DIR##*/}
cd $SLURM_SUBMIT_DIR
echo -n '- List of nodes allocated to the job: '
nodeset -e $SLURM_JOB_NODELIST

# Environment
cd ~
d=$PWD
cd /scratch${PWD#"/prj"}/
module load  anaconda3/2020.11
conda init bash >/dev/null
source $d/.bashrc
conda activate /scratch/app/anaconda3/2020.11
conda activate --stack ./envs
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
cd fft
                                              
# Executable
EXEC="python pc2cs.py"

# Start
echo '$ srun -n' $SLURM_NTASKS ${EXEC##*/}
echo '-- output -----------------------------'
srun -n $SLURM_NTASKS $EXEC
echo '~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'

Overwriting pc2cs.srm


Submit batch script

In [41]:
! sbatch --partition=cpu_dev pc2cs.srm

Submitted batch job 864093


In [42]:
! squeue -n pc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            864061  cpu_small  PD  0:00     1    1
            864062  cpu_small  PD  0:00     1    1
            864093    cpu_dev   R  0:01     1   24


In [44]:
! squeue -n pc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            864061  cpu_small  PD  0:00     1    1
            864062  cpu_small  PD  0:00     1    1


In [45]:
! cat /scratch${PWD#"/prj"}/slurm-864093.out

- Job ID: 864093
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1243
$ srun -n 1 python pc2cs.py
-- output -----------------------------
S: 125.00-0.00j    T: 17.7520 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [47]:
! sbatch --partition=cpu_dev pc2cs.srm

Submitted batch job 864129


In [48]:
! squeue -n pc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            864061  cpu_small  PD  0:00     1    1
            864062  cpu_small  PD  0:00     1    1
            864129    cpu_dev   R  0:02     1   24


In [1]:
! squeue -n pc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            864061  cpu_small  PD  0:00     1    1
            864062  cpu_small  PD  0:00     1    1


In [53]:
! cat /scratch${PWD#"/prj"}/slurm-864129.out

- Job ID: 864129
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1243
$ srun -n 1 python pc2cs.py
-- output -----------------------------
S: 125.00-0.00j    T: 17.8037 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [38]:
! sbatch pc2cs.srm
! sbatch pc2cs.srm

Submitted batch job 864061
Submitted batch job 864062


In [39]:
! squeue -n pc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            864061  cpu_small  PD  0:00     1    1
            864062  cpu_small  PD  0:00     1    1


In [1]:
! squeue -n pc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-864061.out
! cat /scratch${PWD#"/prj"}/slurm-864062.out

- Job ID: 864061
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1301
$ srun -n 1 python pc2cs.py
-- output -----------------------------
S: 125.00-0.00j    T: 18.0248 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 864062
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273
$ srun -n 1 python pc2cs.py
-- output -----------------------------
S: 125.00-0.00j    T: 18.0033 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
